In [21]:
'''
AUTHOR: 
Britney T. Forsyth

DESCRIPTION:
Filters out doublets from sample-level adata using corresponding doublet files from doublet detection. 
Concatenates the sample adata, removes hashtag genes, and applies quality control (QC) metrics to 
filter out low-quality cells based on predefined thresholds, such as mitochondrial fraction and 
ribosomal fraction.
'''

'\nAUTHOR: \nBritney T. Forsyth\n\nDESCRIPTION:\nFilters out doublets from sample-level adata using corresponding doublet files from doublet detection. \nConcatenates the sample adata, removes hashtag genes, and applies quality control (QC) metrics to \nfilter out low-quality cells based on predefined thresholds, such as mitochondrial fraction and \nribosomal fraction.\n'

In [22]:
# Import packages
import os
import pandas as pd
import re
import numpy as np
import glob
from pathlib import Path
from scipy import sparse
from copy import deepcopy
import csv
import itertools
import warnings
import scanpy as sc
from cellbender.remove_background.downstream import anndata_from_h5
import doubletdetection
from sklearn.utils import shuffle
from anndata import AnnData
import anndata

In [3]:
#Look for the .txt files from doublet detection step
# Set the paths
sample_description_path = '/data/chanjlab/forsythb/organoid_analysis_pipeline_scripts/samplematrix_cellbenderpaths_for_doubletdetection.txt'
doublet_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/'

# Read Organoid_Sample_Description.txt
sample_df = pd.read_csv(sample_description_path, sep='\t', index_col=0)
#print(sample_df) 

# Find all doublet files in the doubletdetection directory
doublet_files = glob.glob(os.path.join(doublet_dir, '*doublets.txt'))
#print(doublet_files)

# Create a dictionary to store sample names and corresponding doublet file paths
doublet_dict = {}

# Extract sample names from doublet files and store in the dictionary
for file_path in doublet_files:
    #print(file_path)
    sample_name = os.path.basename(file_path).replace('.doublets.txt', '')
    #print(sample_name)
    doublet_dict[sample_name] = file_path

print(doublet_dict)

{'146P_HISC_shZFP36L2_4': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_HISC_shZFP36L2_4.doublets.txt', 'KG146Li_BASE_shZFP36L2_4': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_BASE_shZFP36L2_4.doublets.txt', 'KG146Li_HISC_shZFP36L2_3': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_HISC_shZFP36L2_3.doublets.txt', '146P_HISC_shCTRL': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_HISC_shCTRL.doublets.txt', '146P_BASE_shCTRL': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_BASE_shCTRL.doublets.txt', '146P_BASE_shZFP36L2_3': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_BASE_shZFP36L2_3.doublets.txt', 'KG146Li_BASE_shCtrl': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_BASE_shCtrl.doublets.txt', 'KG146Li_HISC_shCtrl': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output

In [23]:
# We want to remove all of the ZFPKD replicates called "*_shZFP36L2_4"

# Set the paths
sample_description_path = '/data/chanjlab/forsythb/organoid_analysis_pipeline_scripts/samplematrix_cellbenderpaths_for_doubletdetection.txt'
doublet_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/'

# Read Organoid_Sample_Description.txt
sample_df = pd.read_csv(sample_description_path, sep='\t', index_col=0)

# Find all doublet files in the doubletdetection directory
doublet_files = glob.glob(os.path.join(doublet_dir, '*doublets.txt'))

# Create a dictionary to store sample names and corresponding doublet file paths
doublet_dict = {}

# Extract sample names from doublet files and store in the dictionary
for file_path in doublet_files:
    sample_name = os.path.basename(file_path).replace('.doublets.txt', '')
    # Exclude files with '_shZFP36L2_4'
    if '_shZFP36L2_4' not in sample_name:
        doublet_dict[sample_name] = file_path

print(doublet_dict)
print(len(doublet_dict))

{'KG146Li_HISC_shZFP36L2_3': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_HISC_shZFP36L2_3.doublets.txt', '146P_HISC_shCTRL': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_HISC_shCTRL.doublets.txt', '146P_BASE_shCTRL': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_BASE_shCTRL.doublets.txt', '146P_BASE_shZFP36L2_3': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_BASE_shZFP36L2_3.doublets.txt', 'KG146Li_BASE_shCtrl': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_BASE_shCtrl.doublets.txt', 'KG146Li_HISC_shCtrl': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_HISC_shCtrl.doublets.txt', '146Li_dedifferentiation_shCtrl': '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146Li_dedifferentiation_shCtrl.doublets.txt', '146P_dedifferentiation_shCtrl': '/data/chanjlab/CRC_ZFP36L2.0920

In [4]:
# Let's see what a doublet.txt file looks like for a sample
sample_file = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146Li_dedifferentiation_shCtrl.doublets.txt'

# Open the file in read mode
with open(sample_file, 'r') as file:
    # Read and print the first 5 lines of the file
    for _ in range(5):
        line = file.readline()
        if not line:
            break  
        print(line.strip())

barcode,doublet,doublet_score
CAATGACCACCAGACC-1,0.0,31.424660548872147
CTCGAGGTCTAGAGCT-1,1.0,36.398371624971915
GTGCACGGTAGCCAGA-1,1.0,33.03626654959931
CTCAATTTCAAACTGC-1,1.0,45.68749910478939


In [6]:
root_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/raw.multiplex_copy/'

# Dictionary to store batch names and corresponding list of sample names
batch_to_sample_mapping = {}

# Traverse the batch directories
for batch_name in os.listdir(root_dir):
    batch_path = os.path.join(root_dir, batch_name)

    if os.path.isdir(batch_path):
        # Inside each batch directory, look for demultiplex directories
        demultiplex_dir = os.path.join(batch_path, 'demultiplex')

        if os.path.isdir(demultiplex_dir):
            # Get the list of sample names within demultiplex
            sample_names = [sample_name for sample_name in os.listdir(demultiplex_dir) if os.path.isdir(os.path.join(demultiplex_dir, sample_name))]

            # Add the list of sample names to the dictionary
            batch_to_sample_mapping[batch_name] = sample_names

# Print the mapping
for batch_name, sample_names in batch_to_sample_mapping.items():
    print(f"Batch Name: {batch_name}")
    for sample_name in sample_names:
        print(f" Sample Name: {sample_name}")

Batch Name: 146Li_HISC_IGO_14730_G_1
 Sample Name: KG146Li_HISC_shCtrl
 Sample Name: KG146Li_HISC_shZFP36L2_3
 Sample Name: KG146Li_HISC_shZFP36L2_4
Batch Name: 146Li_dedifferentiation_IGO_14730_I_1
 Sample Name: 146Li_dedifferentiation_shCtrl
 Sample Name: 146Li_dedifferentiation_shZFP36L2_3
 Sample Name: 146Li_dedifferentiation_shZFP36L2_4
Batch Name: 146P_HISC_IGO_14730_1
 Sample Name: 146P_HISC_shZFP36L2_4
 Sample Name: 146P_HISC_shCTRL
 Sample Name: 146P_HISC_shZFP36L2_3
Batch Name: 146Li_BASE_IGO_14730_G_2
 Sample Name: KG146Li_BASE_shZFP36L2_3
 Sample Name: KG146Li_BASE_shZFP36L2_4
 Sample Name: KG146Li_BASE_shCtrl
Batch Name: KG146P_dedifferentiation_repeat_IGO_15487_1
 Sample Name: 146P_dedifferentiation_shCtrl
 Sample Name: 146P_dedifferentiation_shZFP36L2_3
 Sample Name: 146P_dedifferentiation_shZFP36L2_4
Batch Name: 146P_BASE_IGO_14730_3
 Sample Name: 146P_BASE_shCTRL
 Sample Name: 146P_BASE_shZFP36L2_4
 Sample Name: 146P_BASE_shZFP36L2_3


In [24]:
# Set paths
CB_DIR = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/cellbender/'
doublet_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/'

# List to store processed AnnData objects
adata_list = []

# Loop through batches and filter out doublets in combined.h5 files
for batch_name, sample_names in batch_to_sample_mapping.items():

    # Load combined.h5 file for the batch
    h5_file_path = os.path.join(CB_DIR, batch_name, f'output/cellbender.{batch_name}.combined.h5')
    combined_adata = anndata_from_h5(h5_file_path)

    # Define a singlet barcode
    singlet_barcodes = []
    singlet_sample_barcodes = []

    # Iterate through sample names and filter out doublets
    for sample_name in sample_names:
        # Check if the sample has a corresponding doublet file
        if sample_name in doublet_dict:
            doublet_file_path = doublet_dict[sample_name]
            
            #doublet_file_path = os.path.join(doublet_dir, f"{sample_name}.doublets.txt")

            print(f"Processing sample: {sample_name}, Doublet file: {doublet_file_path}")
        
            # Read doublet information from the text file
            doublet_info = pd.read_csv(doublet_file_path, sep=',')

            # Check if 'doublet' column is present in doublet_info DataFrame
            if 'doublet' in doublet_info.columns:
                # Get barcodes with 'doublet' column value equal to 0
                singlet_barcodes += doublet_info.loc[doublet_info['doublet'] == 0, 'barcode'].tolist()
                singlet_sample_barcodes += (sample_name + '_' + doublet_info.loc[doublet_info['doublet'] == 0, 'barcode']).tolist()
            else:
                print(f"'doublet' column not found in {doublet_file_path}")
        else:
            print(f"No doublets file found for sample: {sample_name}")
            # Remove corresponding barcodes from the batch-level AnnData
            not_prepended_barcodes = [barcode for barcode in combined_adata.obs_names if not barcode.startswith(sample_name + '_')]
            combined_adata = combined_adata[np.isin(combined_adata.obs_names, not_prepended_barcodes), :]

    # Filter out doublets from the combined.h5 file
    if len(singlet_barcodes) > 0:
        filtered_barcodes_mask = np.isin(singlet_barcodes, combined_adata.obs_names)
        filtered_barcodes = np.array(singlet_barcodes)[filtered_barcodes_mask]
        combined_adata = combined_adata[filtered_barcodes, :]
        combined_adata.obs_names = singlet_sample_barcodes[:len(filtered_barcodes)]

    # Append the processed AnnData to the list
    adata_list.append(combined_adata)

# Now adata_list contains processed AnnData objects for all batches

/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Processing sample: KG146Li_HISC_shCtrl, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_HISC_shCtrl.doublets.txt
Processing sample: KG146Li_HISC_shZFP36L2_3, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/KG146Li_HISC_shZFP36L2_3.doublets.txt
No doublets file found for sample: KG146Li_HISC_shZFP36L2_4


/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Processing sample: 146Li_dedifferentiation_shCtrl, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146Li_dedifferentiation_shCtrl.doublets.txt
Processing sample: 146Li_dedifferentiation_shZFP36L2_3, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146Li_dedifferentiation_shZFP36L2_3.doublets.txt
No doublets file found for sample: 146Li_dedifferentiation_shZFP36L2_4


/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


...

/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Processing sample: 146P_dedifferentiation_shCtrl, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_dedifferentiation_shCtrl.doublets.txt
Processing sample: 146P_dedifferentiation_shZFP36L2_3, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_dedifferentiation_shZFP36L2_3.doublets.txt
No doublets file found for sample: 146P_dedifferentiation_shZFP36L2_4


/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Processing sample: 146P_BASE_shCTRL, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_BASE_shCTRL.doublets.txt
No doublets file found for sample: 146P_BASE_shZFP36L2_4
Processing sample: 146P_BASE_shZFP36L2_3, Doublet file: /data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/doubletdetection/146P_BASE_shZFP36L2_3.doublets.txt


/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [25]:
# Concatenate the adata_list
for adata in adata_list:
    adata.var_names_make_unique()
    
combined_adata = sc.concat(adata_list, axis = 0, join = 'outer')

In [26]:
combined_adata

AnnData object with n_obs × n_vars = 41114 × 36610
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency'
    obsm: 'gene_expression_encoding'

In [95]:
combined_adata

AnnData object with n_obs × n_vars = 62616 × 36610
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency'
    obsm: 'gene_expression_encoding'

In [27]:
# Concatenate the adata_list
for adata in adata_list:
    adata.var_names_make_unique()

# Concatenate the adata_list and ensure unique var_names
combined_adata = anndata.concat(adata_list, axis=0, join='outer')
combined_adata.var_names_make_unique()

# Remove barcodes without a sample name prepended
combined_adata = combined_adata[combined_adata.obs_names.str.contains('_')]

# Print the resulting obs_names
print(combined_adata.obs_names)

Index(['KG146Li_HISC_shCtrl_AGATGAAGTTCAATCG-1',
       'KG146Li_HISC_shCtrl_CGTTCTGCAACACAAA-1',
       'KG146Li_HISC_shCtrl_TGTAGACAGGAAGAAC-1',
       'KG146Li_HISC_shCtrl_TACCGGGTCACGGGAA-1',
       'KG146Li_HISC_shCtrl_TAGACTGCAAACCATC-1',
       'KG146Li_HISC_shCtrl_CCTCTAGAGTGGATAT-1',
       'KG146Li_HISC_shCtrl_GCGGAAACAGGTTTAC-1',
       'KG146Li_HISC_shCtrl_CTACCCAGTCCGATCG-1',
       'KG146Li_HISC_shCtrl_CAGCAGCGTTCAACGT-1',
       'KG146Li_HISC_shCtrl_CTCCTCCGTATGTCAC-1',
       ...
       '146P_BASE_shZFP36L2_3_GTTGCGGCATCTAACG-1',
       '146P_BASE_shZFP36L2_3_AGATCCAGTCTCGGGT-1',
       '146P_BASE_shZFP36L2_3_TCTGCCACAACCTAAC-1',
       '146P_BASE_shZFP36L2_3_GCACGGTAGGTTGACG-1',
       '146P_BASE_shZFP36L2_3_TGTACAGGTCAGCTTA-1',
       '146P_BASE_shZFP36L2_3_ACATCGAGTGTTTCTT-1',
       '146P_BASE_shZFP36L2_3_TGAGTCATCAAAGCCT-1',
       '146P_BASE_shZFP36L2_3_AGTCAACCATATGCGT-1',
       '146P_BASE_shZFP36L2_3_AAGACAAAGGGCAAGG-1',
       '146P_BASE_shZFP36L2_3_AGGGCTCTCC

In [11]:
# # Concatenate all processed AnnData objects
# if adata_list:
#     #combined_adata = anndata.concat(adata_list, join='outer')

#     # Shuffle the indices
#     shuffled_indices = np.arange(combined_adata.shape[0])
#     np.random.shuffle(shuffled_indices)

#     # Apply the shuffled indices to the combined AnnData object
#     combined_adata = combined_adata[shuffled_indices, :]

#     print("AnnData objects concatenated and indices shuffled successfully.")
# else:
#     print("No processed AnnData objects found for concatenation.")

AnnData objects concatenated and indices shuffled successfully.


In [28]:
# Check if in the var_names, there are HTO- or TSB0- genes
hto_genes = [gene for gene in combined_adata.var_names if "HTO" in gene]
tsb0_genes = [gene for gene in combined_adata.var_names if "TSB0" in gene]

In [29]:
# Remove the HTO- or TSB0- genes
if hto_genes:
    print("Genes containing 'HTO-':", hto_genes)
else:
    print("No genes containing 'HTO-' found.")

if tsb0_genes:
    print("Genes containing 'TSB0':", tsb0_genes)
else:
    print("No genes containing 'TSB0-' found.")

Genes containing 'HTO-': ['CHTOP', 'HTO-1', 'HTO-2', 'HTO-3', 'HTO1', 'HTO2', 'HTO3']
Genes containing 'TSB0': ['TSB0251', 'TSB0252', 'TSB0253']


In [30]:
# Remove HTO- and TSB0- genes from the AnnData
combined_adata = combined_adata[:, ~combined_adata.var_names.isin(hto_genes + tsb0_genes)]

In [31]:
# Check again
hto_genes = [gene for gene in combined_adata.var_names if "HTO" in gene]
tsb0_genes = [gene for gene in combined_adata.var_names if "TSB0" in gene]

if hto_genes:
    print("Genes containing 'HTO-':", hto_genes)
else:
    print("No genes containing 'HTO-' found.")

if tsb0_genes:
    print("Genes containing 'TSB0':", tsb0_genes)
else:
    print("No genes containing 'TSB0-' found.")

No genes containing 'HTO-' found.
No genes containing 'TSB0-' found.


In [33]:
# Calculate qc metrics
sc.pp.calculate_qc_metrics(combined_adata, inplace=True)
combined_adata.obs.loc[:,'log10GenesPerUMI'] =(np.log10(combined_adata.obs.n_genes_by_counts)).div(np.log10(combined_adata.obs.total_counts))
# combined_adata.obs['Sample'] = [re.sub('_[ACGT]+-1$','',i) for i in combined_adata.obs_names]
combined_adata.obs['Sample'] = combined_adata.obs_names.str.replace('_[ACGT]+-1$','',regex=True)

# Calculate total counts
combined_adata.obs['original_total_counts'] = combined_adata.obs['total_counts']
# Calculate library size
combined_adata.obs['log10_original_total_counts'] = np.log10(combined_adata.obs['original_total_counts'])
# Calculate percent of mitochondrial genes
mito_genes = combined_adata.var_names.str.startswith('MT-')
# Compute fraction of mitochondrial genes vs. all genes
combined_adata.obs['mito_frac'] = np.sum(combined_adata[:, mito_genes].X, axis=1) / np.sum(combined_adata.X, axis=1)
# Compute number of ribosomal genes
combined_adata.var['ribo'] = combined_adata.var_names.str.startswith(("RPS", "RPL"))
num_ribosomal_genes = sum(combined_adata.var['ribo'])

mito_frac = pd.Series(combined_adata.obs['mito_frac'], index=combined_adata.obs_names)
libsize = pd.Series(combined_adata.obs['log10_original_total_counts'], index=combined_adata.obs_names)
ngenes = pd.Series(combined_adata.obs['n_genes_by_counts'], index=combined_adata.obs_names)

# Check if 'ribo' is in adata.obs before trying to access it
if 'ribo' in combined_adata.obs.columns:
    ribo_frac = pd.Series(combined_adata.obs['ribo'], index=combined_adata.obs_names)
else:
    ribo_frac = pd.Series(np.nan, index=combined_adata.obs_names)

sc.pp.filter_genes(combined_adata, min_cells=1)
ind1 = combined_adata.obs.total_counts > 500
ind2 = combined_adata.obs.n_genes_by_counts > 400
ind3 = combined_adata.obs.log10GenesPerUMI > 0.8
ind4 = combined_adata.obs.mito_frac < 0.2
ind = ind1.values & ind2.values & ind3.values & ind4.values
combined_adata = combined_adata[ind,:]

/home/forsythb/.local/lib/python3.9/site-packages/scanpy/preprocessing/_qc.py:135: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[obs_metrics.columns] = obs_metrics
/home/forsythb/.local/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/forsythb/.local/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/scratch/lsftmp/6010494.tmpdir/ipykernel_172138/197543707.py:14: RuntimeWarning: invalid value encountered in divide
  combined_adata.obs['mito_frac'] = np.sum(combined_adata[:, mito_genes].X, axis=1) / np.sum(combined_adata.X, axis=1)


In [34]:
combined_adata

View of AnnData object with n_obs × n_vars = 30107 × 31221
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'Sample', 'original_total_counts', 'log10_original_total_counts', 'mito_frac'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells'
    obsm: 'gene_expression_encoding'

In [14]:
combined_adata

View of AnnData object with n_obs × n_vars = 46483 × 31889
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'Sample', 'original_total_counts', 'log10_original_total_counts', 'mito_frac'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells'
    obsm: 'gene_expression_encoding'

In [35]:
# Save the combined adata object
#combined_adata.write_h5ad(os.path.join(out_dir, 'combined_adata.h5'))
out_dir = f'/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/concatenated/'
os.makedirs(out_dir,exist_ok=True)

combined_adata.write_h5ad(out_dir + f'adata.combined.h5ad')
combined_adata.obs.to_csv(out_dir + f'obs.combined.csv', sep ='\t')

fn = out_dir + f'counts.RNA.combined.barcodes.csv'
with open(fn,'w') as f:
    for i in combined_adata.obs.index:
        f.write(i + '\n')

fn = out_dir + f'counts.RNA.combined.genes.csv'
with open(fn,'w') as f:
    for i in combined_adata.var_names:
        f.write(i + '\n')

from scipy.io import mmwrite
from scipy import sparse

fn = out_dir + f'counts.RNA.combined.mtx'
mmwrite(fn, sparse.csr_matrix(combined_adata.X.astype(int)))

/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1294: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
